# W12 Practice — An MCP Server of Our Own

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week12/W12_practice_mcp_server.ipynb)

**Goal.** Build an MCP server and use it from an agent. In Week 11 the agent used servers that others had built. Here we take a public web API of recipes, write an MCP server in front of it, and give the tools of that server to `create_agent`. A question about what to cook needs several requests to the API, and the agent decides which requests to make. The last part is a discussion of how the tools of a server should be defined.

| Part | What you learn | Main names |
|---|---|---|
| 1. The recipe API | two requests to a web API from Python | `httpx2` |
| 2. The MCP server | a tool as a function that calls the API; the tool definition that the server makes from the function | `MCPServer`, `@mcp.tool` |
| 3. The agent with our server | the tools of our server in an agent | `MCPAdapter`, `create_agent` |
| 4. Discussion | the definition of the tools of a server, presented in class | |

**Cell labels.** The first line of each code cell says what to do with it:

- `# ▶ RUN · example`: read each line, then run.
- `# ▶ RUN · setup`: just run.
- `# ✍️ TYPE · discussion`: write your own code (Part 4).

In the cells that save a file, the label is the second line, because `%%writefile` must be the first.

Parts 1 to 3 take about 30 minutes and are not collected. Do the Week 11 practice first.

## An MCP Server over a Web API

A web API is the way programs use a service: a program sends an HTTP request to an address and gets JSON back. A model cannot use such an API directly. It needs tools: a name, a description, and arguments that it can read, and code that runs when it asks. An MCP server gives a web API this form. Each tool of the server is a function that calls the API and returns its reply. The Korean law server of Week 11 is built this way, in front of the Open API of the Ministry of Government Legislation.

In this notebook there are three programs.

| Program | What it does | Where |
|---|---|---|
| Web API | keeps the recipes and answers HTTP requests | TheMealDB, on the internet |
| MCP server | offers tools to MCP clients; each tool calls the web API | `meal_mcp.py`, in this runtime |
| MCP host | connects to the MCP server and gives its tools to an agent | this notebook |

An MCP server in Python is written in five steps.

| Step | Code |
|---|---|
| 1. Make the server object | `mcp = MCPServer("recipes")` |
| 2. Write one function for each tool | `@mcp.tool()` above the function |
| 3. Call the web API in the function and return the reply | `return httpx2.get(...).text` |
| 4. Run the server | `mcp.run()` |
| 5. Connect a client and look at the tools | `MCPAdapter(PythonStdioTransport("meal_mcp.py"))`, `list_tools()` |

The server makes the definition of a tool from the function, as `@tool` did in Week 10: the name of the function becomes the name of the tool, the type hints become the arguments, and the docstring becomes the description.

The KOSIS server of Week 11 ran on the internet, and the client reached it over HTTP. Our server is a file on the same computer as the notebook. The client starts the file as a program and talks to it through its standard input and output. MCP calls this the **stdio** transport.

The code follows the tutorial *Build an MCP server* of the MCP documentation, which puts a server in front of the weather API of the US National Weather Service.

## 0. Setup

*Do:* run the two cells; paste your API key into the second.

In [ ]:
# ▶ RUN · setup: just run
%pip install -q "langchain[mcp]==1.4.4" "langchain-openai==1.7.0" "langgraph==1.2.14"

In [ ]:
# ▶ RUN · setup: just run
import os
from langchain.chat_models import init_chat_model

OPENAI_API_KEY = "PASTE-YOUR-KEY-HERE"   # paste your key between the quotes
if OPENAI_API_KEY != "PASTE-YOUR-KEY-HERE":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY   # LangChain reads the key from here

MODEL = "openai:gpt-5.4"
model = init_chat_model(MODEL)                     # one model object for the whole notebook

## 1. The Recipe API

TheMealDB is a free database of recipes from many countries, with a web API that needs no sign-up. Two of its requests are enough for this notebook.

| Request | What it returns |
|---|---|
| `GET /filter.php?i=<ingredient>` | the meals that use one ingredient |
| `GET /lookup.php?i=<meal ID>` | the recipe of one meal |

### 1.1 A search by ingredient

*Do:* run the cell. It asks for the meals that use chicken breast and prints each meal of the reply.

In [ ]:
# ▶ RUN · example: read each line, then run
import httpx2                                      # an HTTP client for Python; it came with the packages of the setup

API = "https://www.themealdb.com/api/json/v1/1"    # the address of the API; "1" is the public test key

reply = httpx2.get(API + "/filter.php", params={"i": "chicken breast"})   # GET /filter.php?i=chicken breast
meals = reply.json()["meals"]                      # .json() turns the JSON of the reply into Python lists and dicts

print(len(meals), "meals")
for meal in meals:
    print(meal)

### 1.2 The recipe of one meal

*Do:* run the cell. It asks for the recipe of the first meal of 1.1, by its ID, and prints the reply as the API sent it.

In [ ]:
# ▶ RUN · example: read each line, then run
meal_id = meals[0]["idMeal"]                       # the ID of the first meal of the search
reply = httpx2.get(API + "/lookup.php", params={"i": meal_id})   # GET /lookup.php?i=<meal ID>

print(reply.text)                                  # the JSON text of the reply

**Reading the output.** The search returns only the name, a picture, the ID, and the country of each meal. The ingredients and the instructions of a meal need a second request, with its ID. Its reply has 54 fields: the ingredients and their measures are in twenty numbered pairs, `strIngredient1` to `strIngredient20` and `strMeasure1` to `strMeasure20`, and the pairs that a meal does not use are empty. The API also takes only one ingredient per search. A question such as "what can I cook with chicken breast and garlic" therefore needs several requests: one search for each ingredient, then one request for each meal that is chosen. In Part 3 the agent makes these requests itself.

## 2. The MCP Server

The server below has one tool for each of the two requests, written in steps 1 to 4 of the table at the top. Each tool returns the reply of the API as it comes.

### 2.1 The server file

*Do:* read the file, then run the cell. The first line, `%%writefile`, saves the rest of the cell as `meal_mcp.py`. Nothing runs yet.

In [ ]:
%%writefile meal_mcp.py
# ▶ RUN · example: read each line, then run
import httpx2
from mcp.server import MCPServer

API = "https://www.themealdb.com/api/json/v1/1"    # the address of the web API

## Step 1 - the server object
mcp = MCPServer("recipes", log_level="WARNING")    # the name that the server gives to its clients; log only warnings


## Steps 2 and 3 - one function for each tool; the function calls the web API
@mcp.tool()
def search_by_ingredient(ingredient: str) -> str:
    """Find the meals that use one ingredient. Returns the name and the ID of each meal.

    Args:
        ingredient: One ingredient, in English, for example "chicken breast".
    """
    return httpx2.get(API + "/filter.php", params={"i": ingredient}).text   # the reply of the API; this is the tool result


@mcp.tool()
def get_recipe(meal_id: str) -> str:
    """Get the recipe of one meal: its ingredients with their measures, and the instructions.

    Args:
        meal_id: The ID of the meal, for example "52772".
    """
    return httpx2.get(API + "/lookup.php", params={"i": meal_id}).text


## Step 4 - run the server
if __name__ == "__main__":                         # true when this file is started as a program
    mcp.run()                                      # stdio: read requests from standard input, write replies to standard output

### 2.2 The tools of the server

*Do:* run the cell. The client starts `meal_mcp.py` as a program, asks it for its tools, and prints the name, the arguments, and the description of each tool. These definitions are what the model will read.

In [ ]:
# ▶ RUN · example: read each line, then run
from pathlib import Path
from fastmcp.client.transports import PythonStdioTransport
from langchain.mcp import MCPAdapter               # this import prints one warning: this part of LangChain is in beta

## Step 5 - connect a client and look at the tools
server = PythonStdioTransport("meal_mcp.py", log_file=Path("meal_mcp.log"))   # start meal_mcp.py as a program and talk to it over stdio;
                                                   # the messages of the program go to meal_mcp.log (Colab has no place to show them)
async with MCPAdapter(server) as adapter:          # connect; the connection closes at the end of the block
    meal_tools = await adapter.list_tools()        # ask the server for its tools; await waits for the reply

for tool in meal_tools:
    print(tool.name, list(tool.args))              # the name of the tool, then the names of its arguments
    print("description:", tool.description)        # the docstring of the function in meal_mcp.py
    print()

## 3. The Agent with Our Server

The tools go to `create_agent` as in Week 11. Each time the model asks for a tool, the client starts the server again, sends it the call, and the function of the server sends its request to the web API.

The function `show` prints the path of a run: each tool call, the length of each tool result, and the answer.

*Do:* read `QUESTION`, then run the cell. Read the path: which requests the model made, in which order, and how it used the reply of one request in the next.

In [ ]:
# ▶ RUN · example: read each line, then run
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def show(result):
    """Print the path of one run: the tool calls, the length of each tool result, and the answer."""
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:        # one AI message can ask for several tools
                print("TOOL CALL  :", call["name"], call["args"])
        if message.type == "tool":
            print("TOOL RESULT:", message.name, "-", len(message.text), "characters")
    print()
    print(result["messages"][-1].text)             # the last message is the answer

meal_agent = create_agent(model, meal_tools)       # the tools of our server, like any other tools

QUESTION = "닭가슴살과 마늘이 있다. 이 두 재료가 모두 들어가는 요리 두 개를 골라 재료와 조리 순서를 알려줘."
result = await meal_agent.ainvoke({"messages": [HumanMessage(QUESTION)]})   # the tools are asynchronous, so the agent runs with ainvoke
show(result)

**Reading the output.** In our run the model made four requests in three rounds. It searched for chicken breast, then for garlic, because the API takes one ingredient per search. It chose two meals that were in both replies and asked for their two recipes in one reply; the IDs in these calls came from the replies of the searches. Each step was decided by the model from the replies before it: the code of the server only passes each request to the API. The model also wrote the ingredients in English, as the description of the tool asks, although the question was in Korean.

## 4. Discussion: The Tools of a Server

Our server has one tool for each request of the API. This is the fastest way to build a server, and Part 3 showed that it works. But a request of an API is designed for a program, and a tool is read by a model. Anthropic's guide for writers of tools names tools that only wrap the endpoints of an API as a common error. The table puts the points of the guide as five questions and shows what our server does now.

| Question | Our server now |
|---|---|
| Does one tool do one task of the user, or one request of the API? | To find meals with two ingredients, the model searched for each ingredient and compared the two lists itself: two searches before the first recipe in Part 3. |
| Can the model fill the arguments from the words of the user? | The user writes 닭가슴살; the API needs an English name. In Part 3 the model wrote "chicken breast", the example of the description. |
| Does the result hold only what the next step needs? | The search for garlic in Part 3 returned 223 meals, about 42,000 characters with a picture link for each, and the model used two of them. A recipe comes back with all 54 fields, the empty ones too. |
| Can the model read an error and correct its call? | An ingredient that the API does not know returns `{"meals":null}`. |
| Does the description say when to use the tool and what each argument means? | Each description has one sentence and an example of the argument. |

Today's discussion task is to define the tools of the recipe server again and to present your definition in class. First write down the tools that you would offer. For each tool, write the name, the description, the arguments, what it returns, and what it returns when nothing is found. Then build one of them. In the presentation, show your definitions and the path of a run with your tools, next to the path of Part 3. Other requests of the API, such as a search by the name of a meal or by its country, are listed at [themealdb.com/api.php](https://www.themealdb.com/api.php).

*Do:* copy the cell of 2.1 into the first cell below, with `%%writefile meal_mcp.py` as its first line, change or add a tool, and run it. The second cell connects to your server and prints its tools. Add your agent and your question below those lines.

The five questions: Anthropic, *Writing effective tools for agents* (2025). A server of your own, step by step: MCP documentation, [*Build an MCP server*](https://modelcontextprotocol.io/docs/develop/build-server).

In [ ]:
# ✍️ TYPE · discussion: write your own code
# your server file: the cell of 2.1, with your tool

In [ ]:
# ✍️ TYPE · discussion: write your own code
my_server = PythonStdioTransport("meal_mcp.py", log_file=Path("meal_mcp.log"))   # a new program, started from your file
async with MCPAdapter(my_server) as adapter:
    my_tools = await adapter.list_tools()

for tool in my_tools:
    print(tool.name, list(tool.args))

# your agent and your question